# SETUP FOR BETTER GPU

1) Install UCSD VPN: https://blink.ucsd.edu/technology/network/connections/off-campus/VPN/
2) ssh on local machine to ssh username@dsmlp-login.ucsd.edu where username is <username>@ucsd.edu
3) Run the following command on the ssh: launch.sh -c 8 -m 16 -g 1 -v a30
4) Open the link it provides
5) Make sure to **shut down** the pod when you're done from File > Shut Down. And then exit ssh from your terminal

6) If it says GPU already in use (1 of 1 allocated): run the following "kubectl get pods" and "kubectl delete pod <paste-pod-name-here>"


## Starter Code for Datahub

### IF THIS IS YOUR FIRST TIME RUNNING THIS ON DATA HUB
You need to run the cell immediately below so that you can install/setup the kernel. Its called "Python (cse151b)". You should **always** select this kernel in the top right of jupyter. 

### If you're coming back, make sure you re-run everything from the next cell (imports) and further down

The purpose of this starter code is so that you can basically duplicate this and run your experiments in a separate notebook.

# START HERE IF ITS YOUR FIRST TIME HERE

In [1]:
# # =========================================================
# # DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# # =========================================================

# # remove old kernel
# !jupyter kernelspec uninstall -y cse151b

# # Remove old .venv
# !rm -rf .venv

# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# print("Old environment and kernel destroyed.")

# # Create a virtual environment
# !~/.local/bin/uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

# START HERE IF YOU'VE ALREADY SETUP! 

# MAKE SURE TO RESTART KERNEL BEFORE CONTINUING!

## Library Setup

In [2]:
!source ./.venv/bin/activate

import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

## Config + Data Loading
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0" 
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/datahub_experiments.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## Hyperparam Sandbox + LLM Setup

In [4]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     enforce_eager=True,
#     gpu_memory_utilization=0.85,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model and parameters loaded successfully in Eager Mode.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16", 
    quantization=None, 
    load_format="auto",
    enable_prefix_caching=True, 
    enforce_eager=False,        
    gpu_memory_utilization=0.90, 
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    presence_penalty=0.0,
    repetition_penalty=1.05, # slight penalty for repetition
)

print("Model loaded with CUDAGraph + BF16.")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

(EngineCore pid=998) INFO 04-29 17:42:19 [default_loader.py:384] Loading weights took 1.27 seconds
(EngineCore pid=998) INFO 04-29 17:42:19 [gpu_model_runner.py:4820] Model loading took 7.61 GiB memory and 12.971503 seconds
(EngineCore pid=998) INFO 04-29 17:42:28 [backends.py:1051] Using cache directory: /tmp/xdg-cache/vllm/torch_compile_cache/0a52bbc01d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=998) INFO 04-29 17:42:28 [backends.py:1111] Dynamo bytecode transform time: 8.49 s


(EngineCore pid=998) [rank0]:W0429 17:42:29.653000 998 torch/_inductor/utils.py:1679] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=998) INFO 04-29 17:42:35 [backends.py:372] Cache the graph of compile range (1, 32768) for later use
(EngineCore pid=998) INFO 04-29 17:42:41 [backends.py:390] Compiling a graph for compile range (1, 32768) takes 12.74 s
(EngineCore pid=998) INFO 04-29 17:42:44 [decorators.py:655] saved AOT compiled function to /tmp/xdg-cache/vllm/torch_compile_cache/torch_aot_compile/d15c75421a6a312ad970389d8f408957e13e36d74541046e262d4a1d480762ee/rank_0_0/model
(EngineCore pid=998) INFO 04-29 17:42:44 [monitor.py:48] torch.compile took 24.04 s in total
(EngineCore pid=998) INFO 04-29 17:42:47 [monitor.py:76] Initial profiling/warmup run took 3.42 s
(EngineCore pid=998) INFO 04-29 17:42:54 [kv_cache_utils.py:829] Overriding num_gpu_blocks=0 with num_gpu_blocks_override=512
(EngineCore pid=998) INFO 04-29 17:42:54 [gpu_model_runner.py:5876] Profiling CUDA graph memory: PIECEWISE=51 (largest=512), FULL=35 (largest=256)
(EngineCore pid=998) INFO 04-29 17:42:55 [gpu_model_runner.py:5955] E

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 21.23it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 26.04it/s]


(EngineCore pid=998) INFO 04-29 17:43:00 [gpu_model_runner.py:6046] Graph capturing finished in 5 secs, took 2.57 GiB
(EngineCore pid=998) INFO 04-29 17:43:00 [gpu_worker.py:597] CUDA graph pool memory: 2.57 GiB (actual), 2.57 GiB (estimated), difference: 0.0 GiB (0.1%).
(EngineCore pid=998) INFO 04-29 17:43:00 [core.py:283] init engine (profile, create kv cache, warmup model) took 40.76 seconds
Model loaded with CUDAGraph + BF16.


## Prompt Engineering Sandbox

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## Inference (first 5 questions)

In [7]:
#TODO: create smaller experiment set in data loading section and run inference on this!
# Build prompts for first 5 entries
prompts = []
for item in data[:5]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions...


Rendering prompts:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


── Response 0 (id=0) ──
Okay, let's see. I need to find the sum of the first 325 positive even whole numbers. Hmm, first, let me recall what the positive even whole numbers are. They start from 2, right? So the first one is 2, the second is 4, the third is 6, and so on. So it's like 2, 4, 6, ..., up to the 325th term. 

I remember that for arithmetic sequences, there's a formula for the sum. Let me confirm: an arithmetic ...

── Response 1 (id=1) ──
Okay, let's try to solve this integral. The problem is the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s squared plus a squared) ds. Hmm, first, I need to check if the integral converges. Since the integrand is even (because s is squared in the denominator), maybe I can compute it from 0 to infinity and double it? Let me recall that the integral of 1/(s² + a²) ds  ...

── Response 2 (id=2) ──
Okay, let's tackle part (a) first. This seems like a Newton's Law of Cooling problem. I remember that Newton's Law 

## Score Responses + Summary

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [6]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")


mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

Scoring:   0%|          | 5/1126 [00:00<00:31, 35.61it/s]

Scoring complete. 5 results.
EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [7]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/datahub_experiments.jsonl
ERROR 04-29 13:14:33 [core_client.py:667] Engine core proc EngineCore died unexpectedly, shutting down client.
